# Arrays and vectorized math with NumPy

Python stage, lesson P10. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** describe an array by its shape and dtype; do arithmetic on a whole array without a loop and recognize broadcasting; filter and count with a boolean mask; use `np.where` as an if-else for every element.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Arrays for this lesson

In [ ]:
# Run this cell as it is. It builds three arrays with one value for each of the 3,287 order lines.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
quantity = order_items["quantity"].to_numpy()
price = order_items["product_id"].map(price_of).to_numpy()
discount = order_items["discount_usd"].to_numpy()
print("Built three arrays of", quantity.shape[0], "order lines: quantity, price and discount.")

## Unit 1: An array holds one type

**Predict first.** What shape and type does `coffee` have? What does the second array hold when one of its three values is text?

In [ ]:
coffee = np.array([3.00, 3.50, 4.50, 4.50, 5.00, 4.75])  # the six coffee prices
print(coffee.shape, coffee.dtype)
print(np.array([3.25, 3.75, "n/a"]))

## Unit 2: Math on a whole array

**Predict first.** The first three order lines are a Croissant (quantity 1, price 3.25), a Cold Brew (1, 4.75) and Espresso (2, 3.00), none discounted. What are their three amounts? What is `price[:3] * 2`? The shape and the total are for you to read.

In [ ]:
amount = quantity * price - discount
print(amount.shape, amount.sum())
print(amount[:3])
print(price[:3] * 2)

The same total with a loop, as in lesson P5.

In [ ]:
total = 0
for q, p, d in zip(quantity, price, discount):
    total = total + (q * p - d)
print(total)

## Unit 3: A mask of True and False

**Predict first.** The quantities of the first eight lines are 1, 1, 2, 1, 1, 2, 1, 2. Which of the first eight values of the mask `quantity >= 2` are `True`? The shape and the count are for you to read.

In [ ]:
multi = quantity >= 2
print(multi[:8])
print(multi.shape, multi.sum())
print(amount[multi].shape)

## Unit 4: np.where, an IF for every value

**Predict first.** A coupon takes 1.00 off every line of 6 dollars or more. The first eight amounts are 3.25, 4.75, 6.00, 3.25, 9.25, 9.50, 3.00 and 6.50. Which become smaller, and what are they after the coupon? The total is for you to read.

In [ ]:
after_coupon = np.where(amount >= 6, amount - 1, amount)
print(after_coupon[:8])
print(after_coupon.sum())

## Unit 5: Check: shape and type

**Predict first.** Write down the three types, then run the cell.

In [ ]:
print(quantity.dtype, price.dtype, multi.dtype)

## Finish it

### Exercise 1: Finish it

**Task:** How many order lines have a quantity of 3 or more? Replace the first `None` with a mask of those lines, then set `answer_1` to the number of `True` values in it.

**Expected:** one whole number.

In [ ]:
many = None  # replace None: a mask that is True for the lines whose quantity is 3 or more
answer_1 = None  # then set answer_1 to the number of True values in many

In [ ]:
check("Exercise 1", "answer_1", 482, traps={1: "Only 1 line has a quantity above 3. A quantity of exactly 3 counts as 3 or more: which comparison includes it?"})

<details>
<summary>Hint</summary>

Unit 3 builds a mask with a comparison. Then `sum()` counts the `True` values. Does "3 or more" include a quantity of exactly 3?

</details>

<details>
<summary>Solution</summary>

```python
many = quantity >= 3
answer_1 = many.sum()
```

The answer is 482. `quantity > 3` finds only 1: the extra-zero line of order 2075, with a quantity of 20.

</details>

## On your own

### Exercise 2: On your own

**Task:** Every order line of 10 dollars or more gets 1.00 off. What is the total of all 3,287 lines after that? Before you start, say in one sentence what kind of number the answer is. Use `amount` from unit 2 and set `answer_2`.

**Expected:** one decimal number.

In [ ]:
# Exercise 2: take 1.00 off every line of 10 dollars or more, then set answer_2 to the new total of all the lines.

In [ ]:
check("Exercise 2", "answer_2", 22251.25, traps={22307.25: "56 lines are worth exactly 10 dollars, and they get the coupon too. Does your comparison include 10?"})

<details>
<summary>Hint</summary>

Unit 4 has the pattern: `np.where(condition, value if True, value if False)`. Which comparison includes a line of exactly 10 dollars?

</details>

<details>
<summary>Solution</summary>

```python
new_amount = np.where(amount >= 10, amount - 1, amount)
answer_2 = new_amount.sum()
```

The answer is 22251.25: the total of 22754.25 minus 1.00 for each of the 503 lines of 10 dollars or more. A test with `>` leaves out the 56 lines of exactly 10 and gives 22307.25.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** A colleague wants the price of two of each bakery product, writes the line below and gets no error, but the answer is not what was meant. Fix the line, then set `answer_3 = list(two_each)`.

**Expected:** a list of three prices.

In [ ]:
bakery = [3.25, 3.50, 3.75]  # Croissant, Blueberry Muffin, Banana Bread
two_each = bakery * 2  # the colleague expects the price of two of each
answer_3 = None  # fix the line above, then set answer_3 = list(two_each)

In [ ]:
check("Exercise 3", "answer_3", [6.5, 7.0, 7.5], traps={(3.25, 3.5, 3.75, 3.25, 3.5, 3.75): "That list holds six prices: * on a Python list repeats the list. What do you need so that * works on every price?"})

<details>
<summary>Hint</summary>

Print `two_each` and count its values. What type is `bakery`, and what does `*` do to that type?

</details>

<details>
<summary>Solution</summary>

```python
two_each = np.array(bakery) * 2
answer_3 = list(two_each)
```

The answer is `[6.5, 7.0, 7.5]`. On a list, `* 2` repeats the list and gives six prices; on an array, `* 2` doubles each value.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")